In [2]:
# --- INSTALL DEPENDENCIES ---
!pip install -q lifelines plotly pandas numpy scikit-learn

import pandas as pd
import numpy as np
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
import plotly.express as px
import plotly.graph_objects as go
from google.colab import display
import IPython

# --- 1. GENERATE SYNTHETIC CLINICAL & GENOMIC COHORT ---
np.random.seed(42)
n_patients = 500

patient_ids = [f"PT-{i:03d}" for i in range(1, n_patients + 1)]
age = np.random.normal(62, 10, n_patients).clip(30, 90)
treatment = np.random.choice(['Standard Chemotherapy', 'Immunotherapy Combo'], size=n_patients, p=[0.55, 0.45])
biomarker_expr = np.random.gamma(shape=2.0, scale=1.5, size=n_patients) # Continuous genomic expression score

# Simulate survival times (Immunotherapy & higher biomarker expression improve survival)
base_hazard = 0.05
treatment_effect = np.where(treatment == 'Immunotherapy Combo', -0.6, 0.0)
biomarker_effect = -0.35 * (biomarker_expr - biomarker_expr.mean())
risk_score = base_hazard + treatment_effect + biomarker_effect + np.random.normal(0, 0.2, n_patients)
survival_times = np.random.exponential(scale=1.0 / np.exp(risk_score) * 24, size=n_patients).clip(1, 60)

# Simulate censoring (dropouts / study end)
censoring_times = np.random.uniform(12, 60, n_patients)
time = np.minimum(survival_times, censoring_times)
event = (survival_times <= censoring_times).astype(int) # 1 = Death/Event, 0 = Censored

df = pd.DataFrame({
    'patient_id': patient_ids,
    'age': age,
    'treatment': treatment,
    'biomarker_expr': biomarker_expr,
    'time': time,
    'event': event
})

# Stratify biomarker expression into High vs Low for Kaplan-Meier plotting
median_bio = df['biomarker_expr'].median()
df['biomarker_group'] = np.where(df['biomarker_expr'] >= median_bio, 'High Biomarker', 'Low Biomarker')

# --- 2. FIT KAPLAN-MEIER SURVIVAL CURVES ---
kmf_high = KaplanMeierFitter()
kmf_low = KaplanMeierFitter()

high_group = df[df['biomarker_group'] == 'High Biomarker']
low_group = df[df['biomarker_group'] == 'Low Biomarker']

kmf_high.fit(high_group['time'], high_group['event'], label='High Biomarker')
kmf_low.fit(low_group['time'], low_group['event'], label='Low Biomarker')

# Log-rank test between groups
lr_results = logrank_test(high_group['time'], low_group['time'], high_group['event'], low_group['event'])
p_value = lr_results.p_value

# --- 3. FIT COX PROPORTIONAL HAZARDS MODEL ---
cox_df = df[['time', 'event', 'age', 'biomarker_expr']].copy()
cox_df['is_immuno'] = (df['treatment'] == 'Immunotherapy Combo').astype(int)
cph = CoxPHFitter()
cph.fit(cox_df, duration_col='time', event_col='event')
summary_df = cph.summary

# --- 4. BUILD PLOTLY VISUALIZATIONS ---
# A. Kaplan-Meier Curve Plot
km_fig = go.Figure()
km_fig.add_trace(go.Scatter(
    x=kmf_high.timeline, y=kmf_high.survival_function_['High Biomarker'],
    mode='lines', name='High Biomarker (Expr >= Median)',
    line=dict(color='#6366f1', width=3)
))
km_fig.add_trace(go.Scatter(
    x=kmf_low.timeline, y=kmf_low.survival_function_['Low Biomarker'],
    mode='lines', name='Low Biomarker (Expr < Median)',
    line=dict(color='#f43f5e', width=3)
))
km_fig.update_layout(
    title=f"Kaplan-Meier Survival Curves by Biomarker Expression (Log-Rank p-value: {p_value:.4f})",
    xaxis_title="Time (Months)",
    yaxis_title="Survival Probability",
    template="plotly_dark",
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    height=450
)

# B. Cox Hazard Ratios Forest Plot
hazard_ratios = np.exp(cph.params_)
conf_intervals = np.exp(cph.confidence_intervals_)
hr_df = pd.DataFrame({
    'Feature': ['Age', 'Biomarker Expression', 'Immunotherapy Treatment'],
    'HR': hazard_ratios.values,
    'Lower': conf_intervals.iloc[:, 0].values,
    'Upper': conf_intervals.iloc[:, 1].values
})

hr_fig = go.Figure()
hr_fig.add_trace(go.Scatter(
    x=hr_df['HR'], y=hr_df['Feature'],
    mode='markers',
    marker=dict(color='#38bdf8', size=10),
    error_x=dict(
        type='data',
        symmetric=False,
        array=hr_df['Upper'] - hr_df['HR'],
        arrayminus=hr_df['HR'] - hr_df['Lower'],
        color='#38bdf8'
    )
))
hr_fig.add_vline(x=1.0, line_dash="dash", line_color="gray")
hr_fig.update_layout(
    title="Cox Proportional Hazards - Hazard Ratios (95% CI)",
    xaxis_title="Hazard Ratio (HR > 1 indicates higher risk)",
    yaxis_title="",
    template="plotly_dark",
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    height=350
)

# --- 5. RENDER INTERACTIVE DASHBOARD HTML ---
html_output = f"""
<!DOCTYPE html>
<html>
<head>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        body {{ background-color: #030712; color: #f3f4f6; font-family: system-ui, sans-serif; padding: 20px; margin: 0; }}
        .header {{ border-bottom: 1px solid #1e293b; padding-bottom: 15px; margin-bottom: 25px; }}
        .title {{ font-size: 22px; font-weight: bold; background: linear-gradient(to right, #818cf8, #f43f5e); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }}
        .subtitle {{ font-size: 12px; color: #94a3b8; margin-top: 4px; }}
        .grid {{ display: grid; grid-template-columns: 1fr; gap: 20px; }}
        .card {{ background-color: #0f172a; border: 1px solid #1e293b; border-radius: 12px; padding: 20px; }}
        .stats-row {{ display: flex; gap: 15px; margin-bottom: 25px; }}
        .stat-box {{ flex: 1; background: #0f172a; border: 1px solid #1e293b; padding: 15px; border-radius: 10px; text-align: center; }}
        .stat-val {{ font-size: 20px; font-weight: bold; color: #38bdf8; }}
        .stat-lbl {{ font-size: 11px; color: #94a3b8; margin-top: 4px; }}
    </style>
</head>
<body>
    <div class="header">
        <div class="title">Clinical Survival & Biomarker Prognostics Platform</div>
        <div class="subtitle">Cohort analysis of {n_patients} patients across clinical covariates and genomic biomarkers</div>
    </div>

    <div class="stats-row">
        <div class="stat-box">
            <div class="stat-val">{n_patients}</div>
            <div class="stat-lbl">Total Patient Cohort</div>
        </div>
        <div class="stat-box">
            <div class="stat-val">{(df['event'].mean()*100):.1f}%</div>
            <div class="stat-lbl">Event Rate (Mortality/Progression)</div>
        </div>
        <div class="stat-box">
            <div class="stat-val">{p_value:.2e}</div>
            <div class="stat-lbl">Log-Rank Test p-value</div>
        </div>
    </div>

    <div class="grid">
        <div class="card" id="km-plot"></div>
        <div class="card" id="hr-plot" style="margin-top: 20px;"></div>
    </div>

    <script>
        var kmData = {km_fig.to_json()};
        Plotly.newPlot('km-plot', kmData.data, kmData.layout, {{responsive: true}});

        var hrData = {hr_fig.to_json()};
        Plotly.newPlot('hr-plot', hrData.data, hrData.layout, {{responsive: true}});
    </script>
</body>
</html>
"""

display.HTML(html_output)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 10.9 MB/s eta 0:00:00


ImportError: cannot import name 'display' from 'google.colab' (/usr/local/lib/python3.13/dist-packages/google/colab/__init__.py)

In [3]:
# --- 1. INSTALL DEPENDENCIES ---
!pip install -q lifelines plotly pandas numpy scikit-learn

import pandas as pd
import numpy as np
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

# --- 2. GENERATE SYNTHETIC CLINICAL & GENOMIC COHORT ---
np.random.seed(42)
n_patients = 500

patient_ids = [f"PT-{i:03d}" for i in range(1, n_patients + 1)]
age = np.random.normal(62, 10, n_patients).clip(30, 90)
treatment = np.random.choice(['Standard Chemotherapy', 'Immunotherapy Combo'], size=n_patients, p=[0.55, 0.45])
biomarker_expr = np.random.gamma(shape=2.0, scale=1.5, size=n_patients)

# Simulate survival times (Immunotherapy & higher biomarker expression improve survival)
base_hazard = 0.05
treatment_effect = np.where(treatment == 'Immunotherapy Combo', -0.6, 0.0)
biomarker_effect = -0.35 * (biomarker_expr - biomarker_expr.mean())
risk_score = base_hazard + treatment_effect + biomarker_effect + np.random.normal(0, 0.2, n_patients)
survival_times = np.random.exponential(scale=1.0 / np.exp(risk_score) * 24, size=n_patients).clip(1, 60)

# Simulate censoring
censoring_times = np.random.uniform(12, 60, n_patients)
time = np.minimum(survival_times, censoring_times)
event = (survival_times <= censoring_times).astype(int)

df = pd.DataFrame({
    'patient_id': patient_ids,
    'age': age,
    'treatment': treatment,
    'biomarker_expr': biomarker_expr,
    'time': time,
    'event': event
})

median_bio = df['biomarker_expr'].median()
df['biomarker_group'] = np.where(df['biomarker_expr'] >= median_bio, 'High Biomarker', 'Low Biomarker')

# --- 3. FIT KAPLAN-MEIER SURVIVAL CURVES ---
kmf_high = KaplanMeierFitter()
kmf_low = KaplanMeierFitter()

high_group = df[df['biomarker_group'] == 'High Biomarker']
low_group = df[df['biomarker_group'] == 'Low Biomarker']

kmf_high.fit(high_group['time'], high_group['event'], label='High Biomarker')
kmf_low.fit(low_group['time'], low_group['event'], label='Low Biomarker')

lr_results = logrank_test(high_group['time'], low_group['time'], high_group['event'], low_group['event'])
p_value = lr_results.p_value

# --- 4. FIT COX PROPORTIONAL HAZARDS MODEL ---
cox_df = df[['time', 'event', 'age', 'biomarker_expr']].copy()
cox_df['is_immuno'] = (df['treatment'] == 'Immunotherapy Combo').astype(int)
cph = CoxPHFitter()
cph.fit(cox_df, duration_col='time', event_col='event')

# --- 5. BUILD PLOTLY VISUALIZATIONS ---
# Kaplan-Meier Curve
km_fig = go.Figure()
km_fig.add_trace(go.Scatter(
    x=kmf_high.timeline, y=kmf_high.survival_function_['High Biomarker'],
    mode='lines', name='High Biomarker (Expr >= Median)',
    line=dict(color='#6366f1', width=3)
))
km_fig.add_trace(go.Scatter(
    x=kmf_low.timeline, y=kmf_low.survival_function_['Low Biomarker'],
    mode='lines', name='Low Biomarker (Expr < Median)',
    line=dict(color='#f43f5e', width=3)
))
km_fig.update_layout(
    title=f"Kaplan-Meier Survival Curves by Biomarker Expression (Log-Rank p-value: {p_value:.4e})",
    xaxis_title="Time (Months)",
    yaxis_title="Survival Probability",
    template="plotly_dark",
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    height=420
)

# Forest Plot
hazard_ratios = np.exp(cph.params_)
conf_intervals = np.exp(cph.confidence_intervals_)
hr_df = pd.DataFrame({
    'Feature': ['Age', 'Biomarker Expression', 'Immunotherapy Treatment'],
    'HR': hazard_ratios.values,
    'Lower': conf_intervals.iloc[:, 0].values,
    'Upper': conf_intervals.iloc[:, 1].values
})

hr_fig = go.Figure()
hr_fig.add_trace(go.Scatter(
    x=hr_df['HR'], y=hr_df['Feature'],
    mode='markers',
    marker=dict(color='#38bdf8', size=12),
    error_x=dict(
        type='data',
        symmetric=False,
        array=hr_df['Upper'] - hr_df['HR'],
        arrayminus=hr_df['HR'] - hr_df['Lower'],
        color='#38bdf8',
        thickness=2
    )
))
hr_fig.add_vline(x=1.0, line_dash="dash", line_color="gray")
hr_fig.update_layout(
    title="Cox Proportional Hazards - Hazard Ratios (95% CI)",
    xaxis_title="Hazard Ratio (HR < 1 indicates protective effect)",
    yaxis_title="",
    template="plotly_dark",
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    height=320
)

# --- 6. RENDER DASHBOARD DIRECTLY IN COLAB ---
header_html = f"""
<div style="background-color: #030712; color: #f3f4f6; font-family: system-ui, sans-serif; padding: 20px; border-radius: 12px; border: 1px solid #1e293b;">
    <div style="border-bottom: 1px solid #1e293b; padding-bottom: 10px; margin-bottom: 20px;">
        <h2 style="margin: 0; font-size: 22px; background: linear-gradient(to right, #818cf8, #f43f5e); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">
            Clinical Survival & Biomarker Prognostics Platform
        </h2>
        <p style="margin: 4px 0 0 0; font-size: 12px; color: #94a3b8;">Cohort analysis of {n_patients} patients across clinical covariates and genomic biomarkers</p>
    </div>
    <div style="display: flex; gap: 15px;">
        <div style="flex: 1; background: #0f172a; border: 1px solid #1e293b; padding: 12px; border-radius: 8px; text-align: center;">
            <div style="font-size: 18px; font-weight: bold; color: #38bdf8;">{n_patients}</div>
            <div style="font-size: 11px; color: #94a3b8;">Total Patient Cohort</div>
        </div>
        <div style="flex: 1; background: #0f172a; border: 1px solid #1e293b; padding: 12px; border-radius: 8px; text-align: center;">
            <div style="font-size: 18px; font-weight: bold; color: #f43f5e;">{(df['event'].mean()*100):.1f}%</div>
            <div style="font-size: 11px; color: #94a3b8;">Event Rate (Mortality/Progression)</div>
        </div>
        <div style="flex: 1; background: #0f172a; border: 1px solid #1e293b; padding: 12px; border-radius: 8px; text-align: center;">
            <div style="font-size: 18px; font-weight: bold; color: #818cf8;">{p_value:.2e}</div>
            <div style="font-size: 11px; color: #94a3b8;">Log-Rank Test p-value</div>
        </div>
    </div>
</div>
"""

display(HTML(header_html))
km_fig.show()
hr_fig.show()